# Optimized Model Training
Trains and tunes Random Forest, XGBoost, CatBoost, and MLP for delayed shipment prediction.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

In [2]:
def find_best_threshold(y_true, y_prob, start=0.2, stop=0.8, step=0.01):
    thresholds = np.arange(start, stop + 1e-9, step)
    scores = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    idx = int(np.argmax(scores))
    return float(thresholds[idx]), float(scores[idx])

def get_metrics(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'precision_delayed': report['1']['precision'],
        'recall_delayed': report['1']['recall'],
        'f1_delayed': report['1']['f1-score'],
        'threshold': thr
    }

In [3]:
# Load data
df_tree = pd.read_csv('model_ready_tree.csv')
df_cat = pd.read_csv('model_ready_catboost.csv')
df_nn = pd.read_csv('model_ready_nn.csv')
TARGET = 'delayed'

In [4]:
# Random Forest (tree dataset)
X = df_tree.drop(columns=[TARGET])
y = df_tree[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_param_dist = {
    'n_estimators': [300, 500, 700, 900],
    'max_depth': [None, 8, 12, 16, 22],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.5, 0.8],
    'class_weight': ['balanced', 'balanced_subsample', None]
}
rf_search = RandomizedSearchCV(rf, rf_param_dist, n_iter=20, scoring='roc_auc', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=1)
rf_search.fit(X_tr, y_tr)
rf_best = rf_search.best_estimator_
rf_val_prob = rf_best.predict_proba(X_val)[:, 1]
rf_thr, _ = find_best_threshold(y_val, rf_val_prob)
rf_test_prob = rf_best.predict_proba(X_test)[:, 1]
rf_metrics = get_metrics(y_test, rf_test_prob, rf_thr)
print('RF best params:', rf_search.best_params_)
rf_metrics

Fitting 3 folds for each of 20 candidates, totalling 60 fits
RF best params: {'n_estimators': 900, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 0.8, 'max_depth': 22, 'class_weight': 'balanced_subsample'}


{'roc_auc': 0.7985670305424751,
 'precision_delayed': 0.457474450025983,
 'recall_delayed': 0.6107770582793709,
 'f1_delayed': 0.5231256808953154,
 'threshold': 0.5700000000000003}

In [5]:
# XGBoost (tree dataset)
from xgboost import XGBClassifier

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pos_weight = neg / max(pos, 1)

xgb = XGBClassifier(objective='binary:logistic', eval_metric='auc', random_state=RANDOM_STATE, n_jobs=-1, scale_pos_weight=scale_pos_weight)
xgb_param_dist = {
    'n_estimators': [300, 500, 700, 900],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0.0, 0.1, 0.3, 0.5],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0]
}
xgb_search = RandomizedSearchCV(xgb, xgb_param_dist, n_iter=20, scoring='roc_auc', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=1)
xgb_search.fit(X_tr, y_tr)
xgb_best = XGBClassifier(**xgb_search.best_params_, objective='binary:logistic', eval_metric='auc', random_state=RANDOM_STATE, n_jobs=-1, scale_pos_weight=scale_pos_weight)
xgb_best.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
xgb_val_prob = xgb_best.predict_proba(X_val)[:, 1]
xgb_thr, _ = find_best_threshold(y_val, xgb_val_prob)
xgb_test_prob = xgb_best.predict_proba(X_test)[:, 1]
xgb_metrics = get_metrics(y_test, xgb_test_prob, xgb_thr)
print('XGB best params:', xgb_search.best_params_)
xgb_metrics

Fitting 3 folds for each of 20 candidates, totalling 60 fits
XGB best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'n_estimators': 700, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.05, 'gamma': 0.0, 'colsample_bytree': 0.8}


{'roc_auc': 0.8062683504136997,
 'precision_delayed': 0.459766220565814,
 'recall_delayed': 0.6276595744680851,
 'f1_delayed': 0.5307519311626088,
 'threshold': 0.5800000000000003}

In [6]:
# CatBoost (catboost dataset)
from catboost import CatBoostClassifier

X = df_cat.drop(columns=[TARGET])
y = df_cat[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)
cat_cols = X_tr.select_dtypes(include=['object', 'category']).columns.tolist()
cat_features = [X_tr.columns.get_loc(c) for c in cat_cols]
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
class_weights = [1.0, neg / max(pos, 1)]

cb_param_dist = {
    'iterations': [400, 700, 1000],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'bagging_temperature': [0, 0.5, 1, 2],
    'random_strength': [0.5, 1, 2, 5],
    'border_count': [64, 128, 254]
}

best_auc = -np.inf
best_params = None
best_model = None
best_val_prob = None

for params in ParameterSampler(cb_param_dist, n_iter=16, random_state=RANDOM_STATE):
    cb = CatBoostClassifier(**params, loss_function='Logloss', eval_metric='AUC', random_seed=RANDOM_STATE, class_weights=class_weights, allow_writing_files=False, verbose=False)
    cb.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_val, y_val), use_best_model=True, early_stopping_rounds=100)
    val_prob = cb.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    if auc > best_auc:
        best_auc = auc
        best_params = params
        best_model = cb
        best_val_prob = val_prob

cb_thr, _ = find_best_threshold(y_val, best_val_prob)
cb_test_prob = best_model.predict_proba(X_test)[:, 1]
cb_metrics = get_metrics(y_test, cb_test_prob, cb_thr)
print('CatBoost best params:', best_params)
cb_metrics

CatBoost best params: {'random_strength': 2, 'learning_rate': 0.08, 'l2_leaf_reg': 3, 'iterations': 700, 'depth': 8, 'border_count': 254, 'bagging_temperature': 0.5}


{'roc_auc': 0.8058834296655764,
 'precision_delayed': 0.45278490441115615,
 'recall_delayed': 0.632631822386679,
 'f1_delayed': 0.527808595822681,
 'threshold': 0.5600000000000003}

In [7]:
# Neural Network / MLP (nn dataset)
X = df_nn.drop(columns=[TARGET])
y = df_nn[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(activation='relu', solver='adam', max_iter=120, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15, random_state=RANDOM_STATE))
])

mlp_param_dist = {
    'mlp__hidden_layer_sizes': [(128, 64), (256, 128), (256, 128, 64), (512, 256)],
    'mlp__alpha': [1e-5, 1e-4, 1e-3, 1e-2],
    'mlp__learning_rate_init': [1e-4, 3e-4, 1e-3, 3e-3],
    'mlp__batch_size': [256, 512, 1024]
}

mlp_search = RandomizedSearchCV(mlp_pipe, mlp_param_dist, n_iter=12, scoring='roc_auc', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=1)
mlp_search.fit(X_tr, y_tr)
mlp_best = mlp_search.best_estimator_
mlp_val_prob = mlp_best.predict_proba(X_val)[:, 1]
mlp_thr, _ = find_best_threshold(y_val, mlp_val_prob)
mlp_test_prob = mlp_best.predict_proba(X_test)[:, 1]
mlp_metrics = get_metrics(y_test, mlp_test_prob, mlp_thr)
print('MLP best params:', mlp_search.best_params_)
mlp_metrics

Fitting 3 folds for each of 12 candidates, totalling 36 fits
MLP best params: {'mlp__learning_rate_init': 0.0001, 'mlp__hidden_layer_sizes': (128, 64), 'mlp__batch_size': 512, 'mlp__alpha': 1e-05}


{'roc_auc': 0.7877756385552127,
 'precision_delayed': 0.42560767920730763,
 'recall_delayed': 0.6357539315448658,
 'f1_delayed': 0.5098766577019382,
 'threshold': 0.25000000000000006}

In [8]:
results = pd.DataFrame([
    {'model': 'RandomForest', **rf_metrics},
    {'model': 'XGBoost', **xgb_metrics},
    {'model': 'CatBoost', **cb_metrics},
    {'model': 'NeuralNet_MLP', **mlp_metrics},
])
results.sort_values(['roc_auc', 'recall_delayed', 'precision_delayed'], ascending=False)

,model,roc_auc,precision_delayed,recall_delayed,f1_delayed,threshold
1,XGBoost,0.806268,0.459766,0.627660,0.530752,0.58
2,CatBoost,0.805883,0.452785,0.632632,0.527809,0.56
0,RandomForest,0.798567,0.457474,0.610777,0.523126,0.57
3,NeuralNet_MLP,0.787776,0.425608,0.635754,0.509877,0.25
